# Vision-Project — Plane 1A: Cloud Colab Training & LiteRT INT8 Quantization

This notebook executes **Plane 1A** end-to-end in Google Colab:
1. Clones `https://github.com/AbhijeetApurv/Vision-Project` and installs the `ml_pipeline/requirements.txt` dependencies.
2. Exports `YOLOv8n` to a **Full-Integer Quantized (INT8) LiteRT (`.tflite`)** model calibrated on `coco8.yaml` (at `320x320` resolution for 30+ FPS mobile execution).
3. Computes the cryptographic **SHA-256** digest and builds both:
   - `ota_server/models/manifest.json` (for Plane 1B OTA delivery)
   - `android_app/app/src/main/assets/manifest.json` + `yolov8n_int8_factory.tflite` (for Plane 2 Tier-1 factory fallback)
4. Runs `verify_litert_model.py` to validate tensor shapes, quantization scale/zero-point parameters, YOLOv8 head decoding, and host inference latency.

In [ ]:
# 1. Clone Vision-Project and install Plane 1A dependencies
!git clone https://github.com/AbhijeetApurv/Vision-Project.git
%cd Vision-Project/ml_pipeline
!pip install -q -r requirements.txt

In [ ]:
# 2. Export YOLOv8n to INT8 LiteRT (.tflite), compute SHA-256, and populate OTA + Android Factory manifests
!python export_and_quantize.py \
    --weights yolov8n.pt \
    --data coco8.yaml \
    --imgsz 320 \
    --version 1.0.0 \
    --output-dir ../ota_server/models \
    --sync-android-assets ../android_app/app/src/main/assets

In [ ]:
# 3. Verify LiteRT tensor shapes, SHA-256 manifest match, and benchmark host latency
!python verify_litert_model.py \
    --model ../ota_server/models/yolov8n_int8_v1_0_0.tflite \
    --manifest ../ota_server/models/manifest.json \
    --runs 100 \
    --warmup 20 \
    --report ../ota_server/models/benchmark_report.json

In [ ]:
# 4. Inspect the generated manifest.json and download artifacts
!cat ../ota_server/models/manifest.json

from google.colab import files
files.download('../ota_server/models/yolov8n_int8_v1_0_0.tflite')
files.download('../ota_server/models/manifest.json')